# OAE850 EUR TRF+ Attainability Proof

This notebook is a scientific evidence package for the `OAE850|EUR|TRF+` D+1 treasury forecast. It is organized as client-facing arguments rather than as a technical kitchen sink. The target question is:

> Given the permitted information set at D, can daily historical aggregates plausibly support an 80% amount-weighted client score at D+1?

The notebook is self-contained and can run in three modes:

- with no external model results, using Gold-derived features and diagnostic ceilings only;
- with external trained model result dataframes through `MODEL_RESULTS = {"model_name": dataframe, ...}`;
- with external quantile result dataframes through `QUANTILE_RESULTS = {"80_lower": df, "80_upper": df, "95_lower": df, "95_upper": df, ...}`.

External model and quantile dataframes must contain these columns:

`CUTOFF_DATE`, `FORECAST_DATE`, `TARGET_AMOUNT`, `PREDICTION`, `ABS_ERROR`, `EVALUATION_CUTOFF`, `TRAIN_CUTOFF_START`, `TRAIN_CUTOFF_END`.

Predictions are clipped at zero only for scoring. Raw predictions remain available for audit.


In [ ]:
from __future__ import annotations

from pathlib import Path
import math
import re
import sys
import warnings

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

try:
    from IPython.display import Markdown, display
    HAS_IPYTHON = True
except Exception:
    HAS_IPYTHON = False

    class Markdown(str):
        pass

    def display(obj):
        if obj.__class__.__name__ == "Figure":
            title = getattr(getattr(obj, "layout", None), "title", None)
            text = getattr(title, "text", None) or "Plotly figure"
            print(f"[Plotly] {text}")
        elif hasattr(obj, "to_string"):
            print(obj.to_string())
        else:
            print(obj)

warnings.filterwarnings("ignore", category=FutureWarning)
pd.set_option("display.max_columns", 140)
pd.set_option("display.width", 180)


def find_repo_root(start: Path | None = None) -> Path:
    current = (start or Path.cwd()).resolve()
    for candidate in [current, *current.parents]:
        if (candidate / "pyproject.toml").exists() and (candidate / "src").exists():
            return candidate
    raise FileNotFoundError("Could not find repository root from current working directory.")


ROOT = find_repo_root()
SRC = ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

TARGET_SEQUENCE_ID = "OAE850|EUR|TRF+"
TARGET_ENTITY = "OAE850"
TARGET_CURRENCY = "EUR"
TARGET_MOVEMENT = "TRF+"
CLIENT_TARGET_SCORE = 80.0

GOLD_PATH = ROOT / "data" / "gold"
BACKTEST_ROOT = ROOT / "data" / "backtests"
CONFIG_PATH = ROOT / "configs" / "backtests" / "loreal_S_E_TRF_tabular.yaml"
RULESET_PATH = ROOT / "configs" / "rulesets" / "loreal_cash_in_v1.yaml"

# External input hooks. Define these variables in a cell above this one, or pass
# them when executing the notebook programmatically.
MODEL_RESULTS = globals().get("MODEL_RESULTS", {})
QUANTILE_RESULTS = globals().get("QUANTILE_RESULTS", {})
EXTRA_PREDICTION_PATHS = globals().get("EXTRA_PREDICTION_PATHS", [])
SHOW_PLOTS = bool(globals().get("SHOW_PLOTS", True))


def show_plot(fig: go.Figure) -> go.Figure:
    if SHOW_PLOTS:
        display(fig)
    return fig


print(f"Repository root: {ROOT}")
print(f"Gold path exists: {GOLD_PATH.exists()}")
print(f"TRF+ config exists: {CONFIG_PATH.exists()}")
print(f"External MODEL_RESULTS supplied: {len(MODEL_RESULTS)}")
print(f"External QUANTILE_RESULTS supplied: {len(QUANTILE_RESULTS)}")


In [ ]:
from cash_flow_forecast.adapters.local.backtests import load_local_backtest_yaml
from cash_flow_forecast.adapters.local.io import read_gold_outputs
from cash_flow_forecast.adapters.local.rules import load_ruleset_from_yaml
from cash_flow_forecast.contracts.builders import DatasetBuildRequest
from cash_flow_forecast.dataset_building import DatasetBuilder

try:
    from sklearn.feature_selection import mutual_info_regression
except Exception:
    mutual_info_regression = None

try:
    from statsmodels.stats.diagnostic import acorr_ljungbox
except Exception:
    acorr_ljungbox = None


REQUIRED_PREDICTION_COLUMNS = {
    "CUTOFF_DATE",
    "FORECAST_DATE",
    "TARGET_AMOUNT",
    "PREDICTION",
    "ABS_ERROR",
    "EVALUATION_CUTOFF",
    "TRAIN_CUTOFF_START",
    "TRAIN_CUTOFF_END",
}
DATE_COLUMNS = ["CUTOFF_DATE", "FORECAST_DATE", "EVALUATION_CUTOFF", "TRAIN_CUTOFF_START", "TRAIN_CUTOFF_END"]


def _as_float_series(values, index: pd.Index | None = None, name: str | None = None) -> pd.Series:
    if isinstance(values, pd.Series):
        out = values.astype("float64")
        if index is not None:
            out = out.reindex(index)
        return out.rename(name)
    if np.isscalar(values):
        if index is None:
            index = pd.RangeIndex(1)
        return pd.Series(float(values), index=index, name=name, dtype="float64")
    return pd.Series(values, index=index, name=name, dtype="float64")


def daily_business_score(y_true, y_pred) -> pd.Series:
    y = _as_float_series(y_true, name="TARGET_AMOUNT").fillna(0.0).clip(lower=0.0)
    p = _as_float_series(y_pred, index=y.index, name="PREDICTION").fillna(0.0).clip(lower=0.0)
    score = pd.Series(100.0, index=y.index, dtype="float64", name="DAILY_BUSINESS_SCORE")
    positive_actual = y.gt(0.0)
    if positive_actual.any():
        y_pos = y.loc[positive_actual]
        p_pos = p.loc[positive_actual]
        numerator = pd.concat([y_pos, p_pos], axis=1).min(axis=1)
        denominator = pd.concat([y_pos, p_pos], axis=1).max(axis=1)
        score.loc[positive_actual] = 100.0 * numerator / denominator.where(denominator.ne(0.0), np.nan)
        score = score.fillna(0.0)
    score.loc[~positive_actual & p.gt(0.0)] = 0.0
    return score.clip(lower=0.0, upper=100.0)


def weighted_business_score(y_true, y_pred) -> float:
    y = _as_float_series(y_true).fillna(0.0).clip(lower=0.0)
    score = daily_business_score(y, y_pred)
    denominator = float(y.sum())
    if denominator <= 0.0:
        return float("nan")
    return float((score * y).sum() / denominator)


def score_prediction_frame(frame: pd.DataFrame, prediction_col: str = "PREDICTION") -> dict[str, float]:
    y = pd.to_numeric(frame["TARGET_AMOUNT"], errors="coerce").fillna(0.0).clip(lower=0.0)
    p = pd.to_numeric(frame[prediction_col], errors="coerce").fillna(0.0)
    daily = daily_business_score(y, p)
    abs_error = (p.clip(lower=0.0) - y).abs()
    score = weighted_business_score(y, p)
    return {
        "WEIGHTED_BUSINESS_SCORE": score,
        "MEAN_DAILY_SCORE": float(daily.mean()) if len(daily) else float("nan"),
        "AMOUNT_WEIGHTED_PENALTY": 100.0 - score,
        "MAE_CLIPPED": float(abs_error.mean()) if len(abs_error) else float("nan"),
        "TOTAL_ACTUAL_AMOUNT": float(y.sum()),
        "ROWS": int(len(frame)),
    }


def infer_run_family(run_name: str) -> str:
    name = run_name.lower()
    if any(token in name for token in ["quantile", "80_upper", "80_lower", "95_upper", "95_lower", "q80", "q95"]):
        return "quantile"
    if "known_amount" in name:
        return "known_d1"
    if any(token in name for token in ["naive", "moving_average"]):
        return "baseline"
    if any(token in name for token in ["arima", "sarima", "tbats", "theta", "croston", "imapa"]):
        return "classical_or_intermittent"
    if any(token in name for token in ["zero_aware", "cascade", "spike"]):
        return "zero_spike_aware"
    if any(token in name for token in ["stack", "ensemble"]):
        return "ensemble"
    if any(token in name for token in ["lightgbm", "xgboost", "lgb", "xgb"]):
        return "tabular_ml"
    return "other"


def normalize_prediction_frame(frame: pd.DataFrame, run_name: str, family: str | None = None) -> pd.DataFrame:
    missing = REQUIRED_PREDICTION_COLUMNS - set(frame.columns)
    if missing:
        raise ValueError(f"{run_name} is missing required columns: {sorted(missing)}")
    out = frame.copy()
    for column in DATE_COLUMNS:
        out[column] = pd.to_datetime(out[column], errors="coerce").dt.normalize()
    out["TARGET_AMOUNT"] = pd.to_numeric(out["TARGET_AMOUNT"], errors="coerce").fillna(0.0)
    out["PREDICTION"] = pd.to_numeric(out["PREDICTION"], errors="coerce").fillna(0.0)
    out["ABS_ERROR"] = pd.to_numeric(out["ABS_ERROR"], errors="coerce").fillna((out["PREDICTION"] - out["TARGET_AMOUNT"]).abs())
    out = out.dropna(subset=["CUTOFF_DATE", "FORECAST_DATE"]).sort_values("FORECAST_DATE").reset_index(drop=True)
    if not (out["FORECAST_DATE"] - out["CUTOFF_DATE"]).eq(pd.Timedelta(days=1)).all():
        raise ValueError(f"{run_name} contains rows where FORECAST_DATE != CUTOFF_DATE + 1 day")
    out["RUN_NAME"] = run_name
    out["FAMILY"] = family or infer_run_family(run_name)
    out["PREDICTION_CLIPPED"] = out["PREDICTION"].clip(lower=0.0)
    out["DAILY_BUSINESS_SCORE"] = daily_business_score(out["TARGET_AMOUNT"], out["PREDICTION"]).values
    out["WEIGHTED_SCORE_POINTS"] = out["TARGET_AMOUNT"].clip(lower=0.0) * out["DAILY_BUSINESS_SCORE"]
    out["WEIGHTED_PENALTY_POINTS"] = out["TARGET_AMOUNT"].clip(lower=0.0) * (100.0 - out["DAILY_BUSINESS_SCORE"])
    return out


assert daily_business_score([100], [100]).iloc[0] == 100.0
assert daily_business_score([100], [50]).iloc[0] == 50.0
assert daily_business_score([100], [200]).iloc[0] == 50.0
assert daily_business_score([0], [0]).iloc[0] == 100.0
assert daily_business_score([0], [10]).iloc[0] == 0.0
assert weighted_business_score([0, 100], [999, 100]) == 100.0


In [ ]:
local_config = load_local_backtest_yaml(CONFIG_PATH)
ruleset = load_ruleset_from_yaml(local_config.ruleset_path if Path(local_config.ruleset_path).exists() else RULESET_PATH)
gold = read_gold_outputs(GOLD_PATH)

evaluation_cutoffs = pd.date_range(local_config.definition.evaluation.cutoff_start, local_config.definition.evaluation.cutoff_end, freq="D")
feature_result = DatasetBuilder().build(
    DatasetBuildRequest(
        gold_outputs=gold,
        ruleset=ruleset,
        dataset=local_config.definition.dataset,
        cutoff_dates=[date.date() for date in evaluation_cutoffs],
        sequence_id=TARGET_SEQUENCE_ID,
        label_as_of_date=None,
    )
)
feature_frame = feature_result.dataframe.copy()
for column in ["CUTOFF_DATE", "FORECAST_DATE"]:
    feature_frame[column] = pd.to_datetime(feature_frame[column]).dt.normalize()
assert (feature_frame["FORECAST_DATE"] - feature_frame["CUTOFF_DATE"]).eq(pd.Timedelta(days=1)).all()

numeric_feature_columns = [
    column
    for column in feature_result.manifest.feature_columns
    if column in feature_frame.columns and pd.api.types.is_numeric_dtype(feature_frame[column])
]


def frame_from_prediction_series(run_name: str, prediction: pd.Series | np.ndarray | float, family: str, base: pd.DataFrame = feature_frame) -> pd.DataFrame:
    pred = _as_float_series(prediction, index=base.index, name="PREDICTION").fillna(0.0)
    out = base[["CUTOFF_DATE", "FORECAST_DATE", "TARGET_AMOUNT"]].copy()
    out["PREDICTION"] = pred.values
    out["ABS_ERROR"] = (out["PREDICTION"] - out["TARGET_AMOUNT"]).abs()
    out["EVALUATION_CUTOFF"] = out["CUTOFF_DATE"]
    out["TRAIN_CUTOFF_START"] = pd.NaT
    out["TRAIN_CUTOFF_END"] = pd.NaT
    return normalize_prediction_frame(out, run_name, family=family)


def build_simple_candidate_frames(frame: pd.DataFrame) -> dict[str, pd.DataFrame]:
    candidates = {
        "simple/zero": pd.Series(0.0, index=frame.index),
        "simple/known_amount_d1": frame.get("KNOWN_AMOUNT_D1", pd.Series(0.0, index=frame.index)),
        "simple/target_lag_1": frame.get("TARGET_LAG_1", pd.Series(0.0, index=frame.index)),
        "simple/target_lag_7": frame.get("TARGET_LAG_7", pd.Series(0.0, index=frame.index)),
        "simple/target_lag_14": frame.get("TARGET_LAG_14", pd.Series(0.0, index=frame.index)),
        "simple/target_lag_28": frame.get("TARGET_LAG_28", pd.Series(0.0, index=frame.index)),
        "simple/rolling_mean_7": frame.get("TARGET_ROLLING_MEAN_7", pd.Series(0.0, index=frame.index)),
        "simple/rolling_mean_28": frame.get("TARGET_ROLLING_MEAN_28", pd.Series(0.0, index=frame.index)),
    }
    return {name: frame_from_prediction_series(name, pred, family="simple") for name, pred in candidates.items()}


simple_frames = build_simple_candidate_frames(feature_frame)
_schema_smoke = next(iter(simple_frames.values()))[sorted(REQUIRED_PREDICTION_COLUMNS)].copy()
assert len(normalize_prediction_frame(_schema_smoke, "schema_smoke_test")) == len(_schema_smoke)

print(f"Loaded `{TARGET_SEQUENCE_ID}` feature frame with {len(feature_frame):,} rows and {len(numeric_feature_columns):,} numeric feature columns.")


In [ ]:
def safe_run_name_from_path(path: Path, root: Path | None = None) -> str:
    try:
        if root is not None:
            return path.parent.relative_to(root).as_posix()
    except ValueError:
        pass
    return path.parent.as_posix().replace("\\", "/")


def discover_saved_trf_predictions(backtest_root: Path = BACKTEST_ROOT) -> dict[str, pd.DataFrame]:
    frames: dict[str, pd.DataFrame] = {}
    sequence_root = backtest_root / "OAE850_EUR" / "TRF_plus"
    if sequence_root.exists():
        for predictions_path in sorted(sequence_root.rglob("predictions.parquet")):
            frames[safe_run_name_from_path(predictions_path, sequence_root)] = pd.read_parquet(predictions_path)
    return frames


def discover_extra_prediction_paths(paths) -> dict[str, pd.DataFrame]:
    frames: dict[str, pd.DataFrame] = {}
    for raw_path in paths or []:
        path = Path(raw_path)
        if path.is_file():
            frames[path.stem] = pd.read_parquet(path)
        elif path.is_dir():
            for predictions_path in sorted(path.rglob("predictions.parquet")):
                frames[safe_run_name_from_path(predictions_path, path)] = pd.read_parquet(predictions_path)
    return frames


def normalize_external_prediction_frames() -> tuple[dict[str, pd.DataFrame], dict[str, pd.DataFrame], pd.DataFrame]:
    raw_model_frames: dict[str, pd.DataFrame] = {}
    raw_model_frames.update(discover_saved_trf_predictions())
    raw_model_frames.update(discover_extra_prediction_paths(EXTRA_PREDICTION_PATHS))
    raw_model_frames.update(MODEL_RESULTS or {})

    raw_quantile_frames = {f"quantile/{name}": frame for name, frame in (QUANTILE_RESULTS or {}).items()}

    model_frames = {str(run_name): normalize_prediction_frame(frame, str(run_name)) for run_name, frame in raw_model_frames.items()}
    quantile_frames = {str(run_name): normalize_prediction_frame(frame, str(run_name), family="quantile") for run_name, frame in raw_quantile_frames.items()}
    combined = {**model_frames, **quantile_frames}
    manifest = pd.DataFrame(
        [
            {
                "RUN_NAME": name,
                "FAMILY": frame["FAMILY"].iloc[0],
                "ROWS": len(frame),
                "FORECAST_START": frame["FORECAST_DATE"].min(),
                "FORECAST_END": frame["FORECAST_DATE"].max(),
                "SOURCE": "external_or_saved",
            }
            for name, frame in combined.items()
        ]
    )
    return combined, quantile_frames, manifest


prediction_frames, quantile_prediction_frames, prediction_manifest = normalize_external_prediction_frames()
for name, frame in quantile_prediction_frames.items():
    assert frame["FAMILY"].eq("quantile").all(), name

if prediction_manifest.empty:
    display(Markdown("No external/saved TRF+ prediction frames are loaded yet. The notebook will still compute simple baselines and diagnostic ceilings."))
else:
    display(Markdown(f"Loaded **{len(prediction_frames):,}** external/saved TRF+ prediction frame(s), including **{len(quantile_prediction_frames):,}** quantile frame(s)."))
    display(prediction_manifest.sort_values(["FAMILY", "RUN_NAME"]))


In [ ]:
def scores_table(frames: dict[str, pd.DataFrame]) -> pd.DataFrame:
    rows = []
    for run_name, frame in frames.items():
        rows.append({"RUN_NAME": run_name, "FAMILY": frame["FAMILY"].iloc[0], **score_prediction_frame(frame)})
    return pd.DataFrame(rows).sort_values("WEIGHTED_BUSINESS_SCORE", ascending=False).reset_index(drop=True) if rows else pd.DataFrame()


def common_forecast_dates(frames: dict[str, pd.DataFrame]) -> pd.DatetimeIndex:
    date_sets = [set(pd.to_datetime(frame["FORECAST_DATE"]).dt.normalize()) for frame in frames.values()]
    if not date_sets:
        return pd.DatetimeIndex([])
    return pd.DatetimeIndex(sorted(set.intersection(*date_sets)))


def align_frames_on_common_dates(frames: dict[str, pd.DataFrame]) -> dict[str, pd.DataFrame]:
    common_dates = common_forecast_dates(frames)
    if common_dates.empty:
        return {}
    aligned = {name: frame.loc[frame["FORECAST_DATE"].isin(common_dates)].sort_values("FORECAST_DATE").reset_index(drop=True) for name, frame in frames.items()}
    assert len({len(frame) for frame in aligned.values()}) == 1
    return aligned


def hindsight_oracle_from_frames(frames: dict[str, pd.DataFrame], label: str) -> pd.DataFrame:
    aligned = align_frames_on_common_dates(frames)
    if not aligned:
        return pd.DataFrame()
    rows = []
    for run_name, frame in aligned.items():
        work = frame.copy()
        work["SOURCE_RUN_NAME"] = run_name
        rows.append(work)
    long = pd.concat(rows, ignore_index=True, sort=False)
    chosen = long.sort_values(["FORECAST_DATE", "DAILY_BUSINESS_SCORE"], ascending=[True, False]).groupby("FORECAST_DATE", as_index=False).head(1)
    chosen = chosen.rename(columns={"SOURCE_RUN_NAME": "ORACLE_CHOSEN_RUN"})
    chosen["RUN_NAME"] = label
    chosen["FAMILY"] = "oracle"
    return normalize_prediction_frame(chosen, label, family="oracle").assign(ORACLE_CHOSEN_RUN=chosen["ORACLE_CHOSEN_RUN"].values)


def add_fva_columns(table: pd.DataFrame, reference_scores: dict[str, float]) -> pd.DataFrame:
    out = table.copy()
    for label, value in reference_scores.items():
        out[f"FVA_VS_{label}"] = out["WEIGHTED_BUSINESS_SCORE"] - value
    return out


def extract_quantile_role(run_name: str) -> tuple[str, str] | None:
    name = run_name.lower().replace("-", "_").replace("/", "_")
    coverage_match = re.search(r"(80|95)", name)
    if not coverage_match:
        return None
    if "lower" in name:
        return coverage_match.group(1), "lower"
    if "upper" in name:
        return coverage_match.group(1), "upper"
    return None


def quantile_interval_coverage_table(frames: dict[str, pd.DataFrame]) -> pd.DataFrame:
    roles = {}
    for run_name, frame in frames.items():
        role = extract_quantile_role(run_name)
        if role:
            roles[role] = (run_name, frame)
    rows = []
    for coverage in sorted({role[0] for role in roles}):
        lower = roles.get((coverage, "lower"))
        upper = roles.get((coverage, "upper"))
        if not lower or not upper:
            continue
        lower_name, lower_frame = lower
        upper_name, upper_frame = upper
        aligned = align_frames_on_common_dates({"lower": lower_frame, "upper": upper_frame})
        if not aligned:
            continue
        lower_frame = aligned["lower"]
        upper_frame = aligned["upper"]
        actual = lower_frame["TARGET_AMOUNT"].clip(lower=0.0)
        lower_pred = lower_frame["PREDICTION"].clip(lower=0.0)
        upper_pred = upper_frame["PREDICTION"].clip(lower=0.0)
        rows.append(
            {
                "COVERAGE_LEVEL": coverage,
                "LOWER_RUN": lower_name,
                "UPPER_RUN": upper_name,
                "ROWS": len(actual),
                "EMPIRICAL_COVERAGE": float(((actual >= lower_pred) & (actual <= upper_pred)).mean()),
                "MEDIAN_INTERVAL_WIDTH": float((upper_pred - lower_pred).median()),
                "MEAN_INTERVAL_WIDTH": float((upper_pred - lower_pred).mean()),
            }
        )
    return pd.DataFrame(rows)


simple_scores = scores_table(simple_frames)
external_scores = scores_table(prediction_frames)
all_deployable_frames = {**simple_frames, **prediction_frames}
all_deployable_scores = scores_table(all_deployable_frames)
model_library_oracle = hindsight_oracle_from_frames(prediction_frames, "oracle/model_library_hindsight") if prediction_frames else pd.DataFrame()
simple_oracle = hindsight_oracle_from_frames(simple_frames, "oracle/simple_candidate_hindsight")
quantile_interval_table = quantile_interval_coverage_table(quantile_prediction_frames)

best_simple_score = float(simple_scores["WEIGHTED_BUSINESS_SCORE"].max())
reference_scores = {
    "ZERO": float(simple_scores.loc[simple_scores["RUN_NAME"].eq("simple/zero"), "WEIGHTED_BUSINESS_SCORE"].iloc[0]),
    "KNOWN_D1": float(simple_scores.loc[simple_scores["RUN_NAME"].eq("simple/known_amount_d1"), "WEIGHTED_BUSINESS_SCORE"].iloc[0]),
    "WEEKLY_LAG7": float(simple_scores.loc[simple_scores["RUN_NAME"].eq("simple/target_lag_7"), "WEIGHTED_BUSINESS_SCORE"].iloc[0]),
    "BEST_SIMPLE": best_simple_score,
}
if not external_scores.empty:
    external_scores = add_fva_columns(external_scores, reference_scores)


## Argument 1: The official weighted score is dominated by large cash days

**How to read this.** The official metric is amount-weighted, so the feasibility of an 80% score is controlled by the largest actual cash days, not by the typical day. If a small fraction of days carries most of the amount, missing those days can dominate the final KPI even when many low-amount days look acceptable. This is why forecast evaluation must match the business loss, a point also emphasized in [forecast evaluation pitfalls and rolling-origin context](https://pmc.ncbi.nlm.nih.gov/articles/PMC9718476/).


In [ ]:
def amount_concentration_table(frame: pd.DataFrame, amount_col: str = "TARGET_AMOUNT") -> pd.DataFrame:
    y = pd.to_numeric(frame[amount_col], errors="coerce").fillna(0.0).clip(lower=0.0)
    ordered = y.sort_values(ascending=False).reset_index(drop=True)
    total = float(ordered.sum())
    rows = []
    for share in [0.01, 0.02, 0.05, 0.10, 0.20]:
        n = max(1, int(round(len(ordered) * share)))
        rows.append({"TOP_DAY_SHARE": share, "DAY_COUNT": n, "ACTUAL_AMOUNT_SHARE": float(ordered.head(n).sum() / total) if total else np.nan})
    return pd.DataFrame(rows)


def target_profile_table(frame: pd.DataFrame) -> pd.DataFrame:
    y = pd.to_numeric(frame["TARGET_AMOUNT"], errors="coerce").fillna(0.0).clip(lower=0.0)
    return pd.DataFrame(
        [
            {
                "ROWS": len(frame),
                "FORECAST_START": frame["FORECAST_DATE"].min(),
                "FORECAST_END": frame["FORECAST_DATE"].max(),
                "ACTIVE_DAY_RATIO": float(y.gt(0.0).mean()),
                "ZERO_DAY_RATIO": float(y.eq(0.0).mean()),
                "TOTAL_ACTUAL_AMOUNT": float(y.sum()),
                "P90_ACTUAL_AMOUNT": float(y.quantile(0.90)),
                "P95_ACTUAL_AMOUNT": float(y.quantile(0.95)),
                "P99_ACTUAL_AMOUNT": float(y.quantile(0.99)),
            }
        ]
    )


target_profile = target_profile_table(feature_frame)
amount_concentration = amount_concentration_table(feature_frame)
display(target_profile)
display(amount_concentration)

fig = px.bar(
    amount_concentration,
    x="TOP_DAY_SHARE",
    y="ACTUAL_AMOUNT_SHARE",
    text=amount_concentration["ACTUAL_AMOUNT_SHARE"].map(lambda v: f"{v:.1%}"),
    labels={"TOP_DAY_SHARE": "Top forecast-day share", "ACTUAL_AMOUNT_SHARE": "Share of total actual amount"},
    title="TRF+ amount concentration: large days dominate the weighted KPI",
)
fig.update_layout(yaxis_tickformat=".0%", xaxis_tickformat=".0%")
show_plot(fig)


## Argument 2: D+1 known operational information is almost absent

**How to read this.** The project allows daily historical aggregates and point-in-time known movements. For treasury, the decisive question is often whether tomorrow's cash is already known at the decision cutoff. If `KNOWN_AMOUNT_D1` is nearly always zero for TRF+, then the notebook is mostly testing historical pattern recognition, not operational visibility. In statistical terms, this is an information-set limitation: the attainable predictor is bounded by what is known at D, consistent with the Bayes-error view of predictability in [time series predictability and Bayes error](https://arxiv.org/abs/2208.02559).

This argument supports a practical conclusion: reaching 80% may require different data, such as payment run schedules, AP/AR due dates, settlement status, or treasury transfer instructions, rather than another algorithm.


In [ ]:
def trf_known_at_cutoff_daily(gold_outputs, forecast_dates: pd.Series) -> pd.DataFrame:
    forecast_dates = pd.to_datetime(forecast_dates).dt.normalize().drop_duplicates().sort_values()
    base = pd.DataFrame({"VALUE_DATE": forecast_dates})
    realized = gold_outputs.realized_cash_in.copy()
    known = gold_outputs.known_movements_daily.copy()
    for table in [realized, known]:
        for column in ["VALUE_DATE", "TRADE_DATE"]:
            if column in table.columns:
                table[column] = pd.to_datetime(table[column]).dt.normalize()
    realized = realized.loc[realized["SEQUENCE_ID"].astype(str).eq(TARGET_SEQUENCE_ID)]
    realized_daily = realized.groupby("VALUE_DATE", dropna=False, observed=True)["TARGET_AMOUNT"].sum().reset_index()
    daily = base.merge(realized_daily, on="VALUE_DATE", how="left")
    daily["TARGET_AMOUNT"] = daily["TARGET_AMOUNT"].fillna(0.0)
    daily["CUTOFF_DATE"] = daily["VALUE_DATE"] - pd.Timedelta(days=1)
    known = known.loc[known["SEQUENCE_ID"].astype(str).eq(TARGET_SEQUENCE_ID)].copy()
    if known.empty:
        for col in ["KNOWN_AT_CUTOFF_AMOUNT", "LATE_AFTER_CUTOFF_AMOUNT", "KNOWN_AT_CUTOFF_COUNT", "LATE_AFTER_CUTOFF_COUNT"]:
            daily[col] = 0.0
        daily["LATE_AFTER_CUTOFF_AMOUNT"] = daily["TARGET_AMOUNT"]
        return daily
    known["CUTOFF_DATE"] = known["VALUE_DATE"] - pd.Timedelta(days=1)
    known["KNOWN_AT_CUTOFF_AMOUNT"] = known["KNOWN_AMOUNT"].where(known["TRADE_DATE"].le(known["CUTOFF_DATE"]), 0.0)
    known["LATE_AFTER_CUTOFF_AMOUNT"] = known["KNOWN_AMOUNT"].where(known["TRADE_DATE"].gt(known["CUTOFF_DATE"]), 0.0)
    known["KNOWN_AT_CUTOFF_COUNT"] = known["KNOWN_COUNT"].where(known["TRADE_DATE"].le(known["CUTOFF_DATE"]), 0.0)
    known["LATE_AFTER_CUTOFF_COUNT"] = known["KNOWN_COUNT"].where(known["TRADE_DATE"].gt(known["CUTOFF_DATE"]), 0.0)
    known_daily = known.groupby("VALUE_DATE", dropna=False, observed=True).agg(
        KNOWN_AT_CUTOFF_AMOUNT=("KNOWN_AT_CUTOFF_AMOUNT", "sum"),
        LATE_AFTER_CUTOFF_AMOUNT=("LATE_AFTER_CUTOFF_AMOUNT", "sum"),
        KNOWN_AT_CUTOFF_COUNT=("KNOWN_AT_CUTOFF_COUNT", "sum"),
        LATE_AFTER_CUTOFF_COUNT=("LATE_AFTER_CUTOFF_COUNT", "sum"),
    ).reset_index()
    daily = daily.merge(known_daily, on="VALUE_DATE", how="left")
    for column in ["KNOWN_AT_CUTOFF_AMOUNT", "LATE_AFTER_CUTOFF_AMOUNT", "KNOWN_AT_CUTOFF_COUNT", "LATE_AFTER_CUTOFF_COUNT"]:
        daily[column] = daily[column].fillna(0.0)
    return daily


known_daily = trf_known_at_cutoff_daily(gold, feature_frame["FORECAST_DATE"])
known_summary = pd.DataFrame(
    [
        {
            "KNOWN_ABS_COVERAGE_RATIO": known_daily["KNOWN_AT_CUTOFF_AMOUNT"].abs().sum() / known_daily["TARGET_AMOUNT"].clip(lower=0.0).sum(),
            "LATE_ABS_RATIO": known_daily["LATE_AFTER_CUTOFF_AMOUNT"].abs().sum() / known_daily["TARGET_AMOUNT"].clip(lower=0.0).sum(),
            "DAYS_WITH_KNOWN_D1_AMOUNT": int(known_daily["KNOWN_AT_CUTOFF_AMOUNT"].ne(0.0).sum()),
            "DAYS_WITH_KNOWN_D1_AMOUNT_RATIO": float(known_daily["KNOWN_AT_CUTOFF_AMOUNT"].ne(0.0).mean()),
            "KNOWN_AMOUNT_D1_CLIENT_SCORE": weighted_business_score(known_daily["TARGET_AMOUNT"], known_daily["KNOWN_AT_CUTOFF_AMOUNT"]),
        }
    ]
)
display(known_summary)

plot_daily = known_daily.melt(id_vars="VALUE_DATE", value_vars=["TARGET_AMOUNT", "KNOWN_AT_CUTOFF_AMOUNT"], var_name="SERIES", value_name="AMOUNT")
fig = px.line(plot_daily, x="VALUE_DATE", y="AMOUNT", color="SERIES", title="TRF+ realized amount versus amount known by the D+1 cutoff")
show_plot(fig)


## Argument 3: Historical aggregates contain weekly signal, but limited incremental information

**How to read this.** Weekly lags and weekday features can be genuinely informative. The key question is whether that signal is strong enough to identify the large future amounts that drive the weighted score. Mutual information is used here as a nonlinear dependence diagnostic and compared with block-shuffled nulls so weekly/monthly structure is not overstated. The same principle appears in the conditional-expectation view of prediction: only variation explained by the available conditioning variables is reducible; see [conditional expectations and smoothing](https://rafalab.dfci.harvard.edu/dsbook-part-2/ml/conditionals-and-smoothing.html).


In [ ]:
def feature_signal_table(frame: pd.DataFrame, features: list[str]) -> pd.DataFrame:
    y = pd.to_numeric(frame["TARGET_AMOUNT"], errors="coerce").fillna(0.0).clip(lower=0.0)
    rows = []
    for column in features:
        x = pd.to_numeric(frame[column], errors="coerce").fillna(0.0)
        rows.append(
            {
                "FEATURE": column,
                "NON_ZERO_RATIO": float(x.ne(0.0).mean()),
                "PEARSON_WITH_AMOUNT": float(x.corr(y, method="pearson")) if x.nunique() > 1 else np.nan,
                "SPEARMAN_WITH_AMOUNT": float(x.corr(y, method="spearman")) if x.nunique() > 1 else np.nan,
            }
        )
    return pd.DataFrame(rows).sort_values("SPEARMAN_WITH_AMOUNT", key=lambda s: s.abs(), ascending=False).reset_index(drop=True)


def block_shuffle_series(series: pd.Series, block_size: int = 28, seed: int = 0) -> pd.Series:
    rng = np.random.default_rng(seed)
    blocks = [series.iloc[start:start + block_size].copy() for start in range(0, len(series), block_size)]
    order = rng.permutation(len(blocks))
    shuffled = pd.concat([blocks[i] for i in order], ignore_index=True)
    shuffled.index = series.index
    return shuffled


def mutual_information_table(frame: pd.DataFrame, target_col: str = "TARGET_AMOUNT", n_null: int = 10) -> pd.DataFrame:
    if mutual_info_regression is None:
        return pd.DataFrame({"MESSAGE": ["scikit-learn mutual information is not available"]})
    X = frame[numeric_feature_columns].apply(pd.to_numeric, errors="coerce").fillna(0.0)
    y = pd.to_numeric(frame[target_col], errors="coerce").fillna(0.0).clip(lower=0.0)
    actual = mutual_info_regression(X, y, random_state=0, n_neighbors=5)
    null_values = []
    for seed in range(n_null):
        shuffled_y = block_shuffle_series(y, seed=seed)
        null_values.append(mutual_info_regression(X, shuffled_y, random_state=seed, n_neighbors=5))
    null = np.vstack(null_values)
    return pd.DataFrame({"FEATURE": numeric_feature_columns, "MI_AMOUNT": actual, "BLOCK_SHUFFLED_NULL_MEAN": null.mean(axis=0), "MI_MINUS_NULL": actual - null.mean(axis=0)}).sort_values("MI_AMOUNT", ascending=False).reset_index(drop=True)


feature_signal = feature_signal_table(feature_frame, numeric_feature_columns)
feature_mi = mutual_information_table(feature_frame)
display(feature_signal)
display(feature_mi)

fig = px.bar(feature_mi.head(15), x="FEATURE", y=["MI_AMOUNT", "BLOCK_SHUFFLED_NULL_MEAN"], barmode="group", title="Feature mutual information versus block-shuffled null")
fig.update_layout(xaxis_tickangle=-35)
show_plot(fig)


## Argument 4: Models, quantiles, ensembles, and hindsight selection saturate below target

**How to read this.** A weak argument is “we tried many models.” A stronger argument is that deployable models, quantile variants, ensembles, and even a non-deployable hindsight selector all saturate below 80% on the same common forecast dates.

The model comparison uses rolling-origin backtest outputs, aligned on common forecast dates where needed. This follows standard time-series evaluation practice: [Forecasting: Principles and Practice, time-series cross-validation](https://otexts.com/fpp3/tscv.html). Forecast Value Added is included because it is a recognized way to quantify whether additional forecasting steps improve accuracy; see the IJF article page on [Forecast value added in demand planning](https://www.research.lancs.ac.uk/portal/en/publications/forecast-value-added-in-demand-planning%287426ad35-95a9-4272-8f99-450609f3c0fa%29.html).


In [ ]:
display(Markdown("### Simple candidate scores"))
display(simple_scores)

if external_scores.empty:
    display(Markdown("No external/saved trained model results are loaded yet. When `MODEL_RESULTS` or `QUANTILE_RESULTS` are supplied, they will appear here and be used in the model-library oracle."))
else:
    display(Markdown("### External/saved model scores and Forecast Value Added"))
    display(external_scores)

if not quantile_interval_table.empty:
    display(Markdown("### Quantile interval empirical coverage"))
    display(quantile_interval_table)

score_ladder_rows = [
    {"METHOD": "best simple candidate", "TYPE": "deployable/simple", "SCORE": best_simple_score},
    {"METHOD": "simple hindsight oracle", "TYPE": "non-deployable oracle", "SCORE": weighted_business_score(simple_oracle["TARGET_AMOUNT"], simple_oracle["PREDICTION"])},
]
if not external_scores.empty:
    score_ladder_rows.append({"METHOD": f"best external model: {external_scores.iloc[0]['RUN_NAME']}", "TYPE": "deployable/model", "SCORE": float(external_scores.iloc[0]["WEIGHTED_BUSINESS_SCORE"])})
if not model_library_oracle.empty:
    score_ladder_rows.append({"METHOD": "model-library hindsight oracle", "TYPE": "non-deployable oracle", "SCORE": weighted_business_score(model_library_oracle["TARGET_AMOUNT"], model_library_oracle["PREDICTION"])})
score_ladder_rows.append({"METHOD": "client target", "TYPE": "target", "SCORE": CLIENT_TARGET_SCORE})
score_ladder = pd.DataFrame(score_ladder_rows)
display(score_ladder)

fig = px.bar(score_ladder.sort_values("SCORE"), x="SCORE", y="METHOD", color="TYPE", orientation="h", text=score_ladder.sort_values("SCORE")["SCORE"].map(lambda v: f"{v:.1f}%"), title="Attainability ladder: deployable scores, hindsight oracles, and target")
fig.add_vline(x=CLIENT_TARGET_SCORE, line_dash="dash", line_color="red", annotation_text="80% target")
show_plot(fig)

if not external_scores.empty:
    fig = px.bar(external_scores.sort_values("WEIGHTED_BUSINESS_SCORE"), x="WEIGHTED_BUSINESS_SCORE", y="RUN_NAME", color="FAMILY", orientation="h", title="External/saved model saturation curve")
    fig.add_vline(x=CLIENT_TARGET_SCORE, line_dash="dash", line_color="red", annotation_text="80% target")
    show_plot(fig)

if not model_library_oracle.empty:
    display(Markdown("### Model-library hindsight oracle selection counts"))
    display(model_library_oracle["ORACLE_CHOSEN_RUN"].value_counts().rename_axis("RUN_NAME").reset_index(name="SELECTED_DAYS"))


## Argument 5: The analog Bayes-style ceiling remains below the 80% requirement

**How to read this.** The theoretical best predictor for a score is a Bayes action: choose the prediction that maximizes expected score conditional on the information available at D. The true conditional distribution is unknown, so this notebook approximates it with historical analogs and a rich action grid. This is intentionally optimistic compared with a normal production model.

If even this optimistic analog ceiling is materially below 80%, the conclusion is stronger than “the current model failed.” It suggests the information set itself is insufficient. For the conceptual link between predictability and Bayes error, see [Equivalence between Time Series Predictability and Bayes Error Rate](https://arxiv.org/abs/2208.02559).


In [ ]:
def expected_amount_weighted_utilities_np(analog_targets, actions) -> np.ndarray:
    y = np.clip(np.asarray(analog_targets, dtype="float64"), 0.0, None)
    a = np.clip(np.asarray(actions, dtype="float64"), 0.0, None)
    if y.size == 0:
        return np.full(a.shape, np.nan, dtype="float64")
    y_matrix = y[None, :]
    a_matrix = a[:, None]
    denominator = np.maximum(y_matrix, a_matrix)
    numerator = np.minimum(y_matrix, a_matrix)
    scores = np.divide(100.0 * numerator, denominator, out=np.zeros_like(denominator), where=denominator > 0.0)
    scores = np.where((y_matrix <= 0.0) & (a_matrix <= 0.0), 100.0, scores)
    if y.sum() <= 0.0:
        return scores.mean(axis=1)
    return (scores * y_matrix).sum(axis=1) / y.sum()


def model_predictions_by_date(frames: dict[str, pd.DataFrame]) -> dict[pd.Timestamp, list[float]]:
    by_date: dict[pd.Timestamp, list[float]] = {}
    for frame in frames.values():
        for date, group in frame.groupby("FORECAST_DATE"):
            by_date.setdefault(pd.Timestamp(date).normalize(), []).extend(group["PREDICTION"].astype(float).tolist())
    return by_date


def analog_bayes_optimizer(frame: pd.DataFrame, feature_columns: list[str], frames_for_extra_actions: dict[str, pd.DataFrame] | None = None, min_history: int = 180, k_neighbors: int = 60) -> pd.DataFrame:
    work = frame.sort_values("CUTOFF_DATE").reset_index(drop=True).copy()
    X = work[feature_columns].apply(pd.to_numeric, errors="coerce").fillna(0.0).to_numpy(dtype="float64")
    y_all = pd.to_numeric(work["TARGET_AMOUNT"], errors="coerce").fillna(0.0).clip(lower=0.0).to_numpy(dtype="float64")
    extra_by_date = model_predictions_by_date(frames_for_extra_actions or {})
    row_action_columns = [column for column in ["KNOWN_AMOUNT_D1", "TARGET_LAG_1", "TARGET_LAG_7", "TARGET_LAG_14", "TARGET_LAG_28", "TARGET_ROLLING_MEAN_7", "TARGET_ROLLING_MEAN_28"] if column in work.columns]
    row_action_values = work[row_action_columns].apply(pd.to_numeric, errors="coerce").fillna(0.0).to_numpy(dtype="float64") if row_action_columns else np.zeros((len(work), 0))
    forecast_dates = pd.to_datetime(work["FORECAST_DATE"]).dt.normalize().tolist()
    rows = []
    for i in range(len(work)):
        if i < min_history:
            fallback_candidates = row_action_values[i] if row_action_values.shape[1] else np.array([0.0])
            fallback = float(np.nanmedian(np.clip(fallback_candidates, 0.0, None))) if fallback_candidates.size else 0.0
            rows.append({"PREDICTION": max(0.0, fallback), "ANALOG_COUNT": i, "EXPECTED_ANALOG_SCORE": np.nan, "ANALOG_TARGET_VARIANCE": np.nan})
            continue
        X_hist = X[:i]
        mean = X_hist.mean(axis=0)
        std = X_hist.std(axis=0)
        std[std == 0.0] = 1.0
        distances = np.sqrt(np.sum(((X_hist - X[i]) / std) ** 2, axis=1))
        k = min(k_neighbors, len(distances))
        nearest_positions = np.argpartition(distances, k - 1)[:k]
        analog_targets = y_all[nearest_positions]
        history_targets = y_all[:i]
        history_quantiles = np.quantile(history_targets, [0.10, 0.25, 0.50, 0.75, 0.90, 0.95, 0.99])
        recent_targets = history_targets[max(0, i - 28):i]
        extra_actions = np.asarray(extra_by_date.get(forecast_dates[i], []), dtype="float64")
        actions = np.concatenate([np.array([0.0]), row_action_values[i], history_quantiles, recent_targets, extra_actions])
        actions = np.unique(np.clip(actions[np.isfinite(actions)], 0.0, None))
        utilities = expected_amount_weighted_utilities_np(analog_targets, actions)
        best_idx = int(np.nanargmax(utilities))
        rows.append({"PREDICTION": float(actions[best_idx]), "ANALOG_COUNT": int(len(analog_targets)), "EXPECTED_ANALOG_SCORE": float(utilities[best_idx]), "ANALOG_TARGET_VARIANCE": float(np.var(analog_targets, ddof=0))})
    result = work[["CUTOFF_DATE", "FORECAST_DATE", "TARGET_AMOUNT"]].copy()
    result["PREDICTION"] = [row["PREDICTION"] for row in rows]
    result["ABS_ERROR"] = (result["PREDICTION"] - result["TARGET_AMOUNT"]).abs()
    result["EVALUATION_CUTOFF"] = result["CUTOFF_DATE"]
    result["TRAIN_CUTOFF_START"] = pd.NaT
    result["TRAIN_CUTOFF_END"] = pd.NaT
    result = normalize_prediction_frame(result, "diagnostic/analog_bayes_score_optimizer", family="diagnostic_ceiling")
    result["ANALOG_COUNT"] = [row["ANALOG_COUNT"] for row in rows]
    result["EXPECTED_ANALOG_SCORE"] = [row["EXPECTED_ANALOG_SCORE"] for row in rows]
    result["ANALOG_TARGET_VARIANCE"] = [row["ANALOG_TARGET_VARIANCE"] for row in rows]
    return result


def block_bootstrap_score(frame: pd.DataFrame, block_size: int = 28, n_bootstrap: int = 300, seed: int = 0) -> pd.DataFrame:
    rng = np.random.default_rng(seed)
    work = frame.sort_values("FORECAST_DATE").reset_index(drop=True).copy()
    blocks = [work.iloc[start:start + block_size] for start in range(0, len(work), block_size)]
    scores = []
    for _ in range(n_bootstrap):
        sampled = []
        while sum(len(block) for block in sampled) < len(work):
            sampled.append(blocks[int(rng.integers(0, len(blocks)))])
        boot = pd.concat(sampled, ignore_index=True).head(len(work))
        scores.append(weighted_business_score(boot["TARGET_AMOUNT"], boot["PREDICTION"]))
    return pd.DataFrame({"BOOTSTRAP_SCORE": scores})


bayes_feature_columns = [column for column in numeric_feature_columns if feature_frame[column].nunique(dropna=False) > 1]
analog_bayes = analog_bayes_optimizer(feature_frame, bayes_feature_columns, prediction_frames)
analog_bayes_score = weighted_business_score(analog_bayes["TARGET_AMOUNT"], analog_bayes["PREDICTION"])
bootstrap = block_bootstrap_score(analog_bayes)
bootstrap_ci = bootstrap["BOOTSTRAP_SCORE"].quantile([0.025, 0.50, 0.975]).rename(index={0.025: "LOWER_95", 0.5: "MEDIAN", 0.975: "UPPER_95"})
analog_summary = pd.DataFrame(
    [
        {"METHOD": "analog Bayes-style optimizer", **score_prediction_frame(analog_bayes)},
        {"METHOD": "analog Bayes bootstrap lower 95%", "WEIGHTED_BUSINESS_SCORE": float(bootstrap_ci.loc["LOWER_95"])},
        {"METHOD": "analog Bayes bootstrap median", "WEIGHTED_BUSINESS_SCORE": float(bootstrap_ci.loc["MEDIAN"])},
        {"METHOD": "analog Bayes bootstrap upper 95%", "WEIGHTED_BUSINESS_SCORE": float(bootstrap_ci.loc["UPPER_95"])},
        {"METHOD": "client target", "WEIGHTED_BUSINESS_SCORE": CLIENT_TARGET_SCORE},
    ]
)
display(analog_summary)

fig = px.histogram(bootstrap, x="BOOTSTRAP_SCORE", nbins=40, title="Block-bootstrap distribution of the analog Bayes-style score")
fig.add_vline(x=CLIENT_TARGET_SCORE, line_dash="dash", line_color="red", annotation_text="80% target")
show_plot(fig)


## Argument 6: A small set of hard days explains most of the business penalty

**How to read this.** If top penalty days explain most of the score gap, the target is not mainly about average behavior. It is about anticipating exceptional, high-amount days. This section identifies those days and quantifies their contribution to the total weighted penalty. This is a metric-specific complement to standard rolling-origin forecast evaluation, as described in [FPP3 time-series cross-validation](https://otexts.com/fpp3/tscv.html).


In [ ]:
def business_penalty_frame(frame: pd.DataFrame) -> pd.DataFrame:
    return normalize_prediction_frame(frame, frame["RUN_NAME"].iloc[0] if "RUN_NAME" in frame.columns else "penalty_frame", family=frame["FAMILY"].iloc[0] if "FAMILY" in frame.columns else None)


def penalty_concentration_table(frame: pd.DataFrame) -> pd.DataFrame:
    penalty = business_penalty_frame(frame).sort_values("WEIGHTED_PENALTY_POINTS", ascending=False)
    total_penalty = float(penalty["WEIGHTED_PENALTY_POINTS"].sum())
    rows = []
    for share in [0.01, 0.02, 0.05, 0.10, 0.20]:
        n = max(1, int(round(len(penalty) * share)))
        rows.append(
            {
                "TOP_DAY_SHARE": share,
                "DAY_COUNT": n,
                "BUSINESS_PENALTY_SHARE": float(penalty.head(n)["WEIGHTED_PENALTY_POINTS"].sum() / total_penalty) if total_penalty else np.nan,
                "ACTUAL_AMOUNT_SHARE": float(penalty.head(n)["TARGET_AMOUNT"].clip(lower=0.0).sum() / penalty["TARGET_AMOUNT"].clip(lower=0.0).sum()),
            }
        )
    return pd.DataFrame(rows)


def spike_penalty_table(frame: pd.DataFrame) -> pd.DataFrame:
    penalty = business_penalty_frame(frame)
    y = penalty["TARGET_AMOUNT"].clip(lower=0.0)
    total_amount = float(y.sum())
    total_penalty = float(penalty["WEIGHTED_PENALTY_POINTS"].sum())
    rows = []
    for label, threshold in {"Q90": y.quantile(0.90), "Q95": y.quantile(0.95), "Q99": y.quantile(0.99)}.items():
        mask = y.ge(threshold)
        rows.append(
            {
                "SPIKE_DEFINITION": label,
                "THRESHOLD": float(threshold),
                "DAY_SHARE": float(mask.mean()),
                "ACTUAL_AMOUNT_SHARE": float(y.loc[mask].sum() / total_amount) if total_amount else np.nan,
                "BUSINESS_PENALTY_SHARE": float(penalty.loc[mask, "WEIGHTED_PENALTY_POINTS"].sum() / total_penalty) if total_penalty else np.nan,
            }
        )
    return pd.DataFrame(rows)


def select_best_evidence_frame() -> tuple[str, pd.DataFrame]:
    if not external_scores.empty:
        best_name = str(external_scores.iloc[0]["RUN_NAME"])
        return best_name, prediction_frames[best_name]
    return "diagnostic/analog_bayes_score_optimizer", analog_bayes


best_evidence_name, best_evidence_frame = select_best_evidence_frame()
penalty_concentration = penalty_concentration_table(best_evidence_frame)
spike_penalty = spike_penalty_table(best_evidence_frame)
worst_days = business_penalty_frame(best_evidence_frame).sort_values("WEIGHTED_PENALTY_POINTS", ascending=False).head(30)

display(Markdown(f"Penalty concentration shown for `{best_evidence_name}`."))
display(penalty_concentration)
display(spike_penalty)
display(worst_days[["FORECAST_DATE", "TARGET_AMOUNT", "PREDICTION", "DAILY_BUSINESS_SCORE", "WEIGHTED_PENALTY_POINTS"]])

fig = px.bar(penalty_concentration, x="TOP_DAY_SHARE", y="BUSINESS_PENALTY_SHARE", text=penalty_concentration["BUSINESS_PENALTY_SHARE"].map(lambda v: f"{v:.1%}"), title=f"Worst-day business penalty concentration for {best_evidence_name}")
fig.update_layout(xaxis_tickformat=".0%", yaxis_tickformat=".0%")
show_plot(fig)

fig = px.scatter(worst_days, x="FORECAST_DATE", y="WEIGHTED_PENALTY_POINTS", size="TARGET_AMOUNT", color="DAILY_BUSINESS_SCORE", hover_data=["TARGET_AMOUNT", "PREDICTION", "DAILY_BUSINESS_SCORE"], title="Worst TRF+ days by weighted business penalty")
show_plot(fig)


## Argument 7: Removing worst days quantifies how dependent the target is on exceptional events

**How to read this.** This is a diagnostic stress test, not a production scoring rule. If a model only approaches 80% after excluding a small number of high-amount penalty days, the business target depends on exceptional events that were not inferable from the permitted historical aggregates. Treat this as sensitivity analysis around the loss distribution, not as a replacement for the official backtest; see the broader evaluation caveats in [forecast evaluation pitfalls and best practices](https://pmc.ncbi.nlm.nih.gov/articles/PMC9718476/).


In [ ]:
def score_after_removing_dates(frame: pd.DataFrame, removed_dates: set[pd.Timestamp]) -> dict[str, float]:
    work = business_penalty_frame(frame)
    removed_mask = work["FORECAST_DATE"].isin(removed_dates)
    kept = work.loc[~removed_mask]
    removed = work.loc[removed_mask]
    return {
        "score_after_exclusion": weighted_business_score(kept["TARGET_AMOUNT"], kept["PREDICTION"]) if not kept.empty else np.nan,
        "removed_day_share": float(removed_mask.mean()),
        "removed_amount_share": float(removed["TARGET_AMOUNT"].clip(lower=0.0).sum() / work["TARGET_AMOUNT"].clip(lower=0.0).sum()) if work["TARGET_AMOUNT"].clip(lower=0.0).sum() else np.nan,
        "removed_penalty_share": float(removed["WEIGHTED_PENALTY_POINTS"].sum() / work["WEIGHTED_PENALTY_POINTS"].sum()) if work["WEIGHTED_PENALTY_POINTS"].sum() else np.nan,
        "kept_rows": int(len(kept)),
    }


def minimum_removed_to_reach_target(frame: pd.DataFrame, target_score: float = CLIENT_TARGET_SCORE) -> dict[str, float]:
    work = business_penalty_frame(frame).sort_values("WEIGHTED_PENALTY_POINTS", ascending=False).reset_index(drop=True)
    total_amount = float(work["TARGET_AMOUNT"].clip(lower=0.0).sum())
    total_score_points = float(work["WEIGHTED_SCORE_POINTS"].sum())
    total_penalty = float(work["WEIGHTED_PENALTY_POINTS"].sum())
    cum_amount = work["TARGET_AMOUNT"].clip(lower=0.0).cumsum()
    cum_score_points = work["WEIGHTED_SCORE_POINTS"].cumsum()
    cum_penalty = work["WEIGHTED_PENALTY_POINTS"].cumsum()
    for n in range(0, len(work) + 1):
        removed_amount = float(cum_amount.iloc[n - 1]) if n > 0 else 0.0
        removed_score_points = float(cum_score_points.iloc[n - 1]) if n > 0 else 0.0
        removed_penalty = float(cum_penalty.iloc[n - 1]) if n > 0 else 0.0
        remaining_amount = total_amount - removed_amount
        if remaining_amount <= 0.0:
            break
        score = (total_score_points - removed_score_points) / remaining_amount
        if score >= target_score:
            return {
                "minimum_removed_days_to_reach_80": n,
                "minimum_removed_day_share_to_reach_80": n / len(work),
                "minimum_removed_amount_share_to_reach_80": removed_amount / total_amount if total_amount else np.nan,
                "minimum_removed_penalty_share_to_reach_80": removed_penalty / total_penalty if total_penalty else np.nan,
            }
    return {
        "minimum_removed_days_to_reach_80": np.nan,
        "minimum_removed_day_share_to_reach_80": np.nan,
        "minimum_removed_amount_share_to_reach_80": np.nan,
        "minimum_removed_penalty_share_to_reach_80": np.nan,
    }


def per_model_exclusion_curve(frames: dict[str, pd.DataFrame], shares=(0.0, 0.01, 0.02, 0.05, 0.10, 0.20)) -> pd.DataFrame:
    rows = []
    for run_name, frame in frames.items():
        work = business_penalty_frame(frame).sort_values("WEIGHTED_PENALTY_POINTS", ascending=False)
        min_removed = minimum_removed_to_reach_target(frame)
        for share in shares:
            n = int(round(len(work) * share))
            removed_dates = set(work.head(n)["FORECAST_DATE"])
            rows.append({"RUN_NAME": run_name, "EXCLUSION_MODE": "own_worst_days", **score_after_removing_dates(frame, removed_dates), **min_removed})
    return pd.DataFrame(rows)


def shared_date_exclusion_curve(frames: dict[str, pd.DataFrame], reference_frame: pd.DataFrame, shares=(0.0, 0.01, 0.02, 0.05, 0.10, 0.20)) -> pd.DataFrame:
    reference = business_penalty_frame(reference_frame).sort_values("WEIGHTED_PENALTY_POINTS", ascending=False)
    rows = []
    for share in shares:
        n = int(round(len(reference) * share))
        removed_dates = set(reference.head(n)["FORECAST_DATE"])
        for run_name, frame in frames.items():
            rows.append({"RUN_NAME": run_name, "EXCLUSION_MODE": "shared_reference_worst_days", **score_after_removing_dates(frame, removed_dates)})
    return pd.DataFrame(rows)


exclusion_frames = dict(prediction_frames)
if not exclusion_frames:
    best_simple_name = str(simple_scores.iloc[0]["RUN_NAME"])
    exclusion_frames[best_simple_name] = simple_frames[best_simple_name]
exclusion_frames["diagnostic/analog_bayes_score_optimizer"] = analog_bayes

per_model_exclusion = per_model_exclusion_curve(exclusion_frames)
shared_exclusion = shared_date_exclusion_curve(exclusion_frames, best_evidence_frame)
assert shared_exclusion.groupby("RUN_NAME")["removed_day_share"].nunique().ge(1).all()

display(Markdown("### Per-model exclusion curve"))
display(per_model_exclusion)
display(Markdown("### Shared-date exclusion curve"))
display(shared_exclusion)

fig = px.line(per_model_exclusion, x="removed_day_share", y="score_after_exclusion", color="RUN_NAME", markers=True, title="Per-model score after excluding each model's own worst days")
fig.add_hline(y=CLIENT_TARGET_SCORE, line_dash="dash", line_color="red", annotation_text="80% target")
fig.update_layout(xaxis_tickformat=".0%")
show_plot(fig)

fig = px.line(shared_exclusion, x="removed_day_share", y="score_after_exclusion", color="RUN_NAME", markers=True, title=f"Scores after excluding shared worst days from {best_evidence_name}")
fig.add_hline(y=CLIENT_TARGET_SCORE, line_dash="dash", line_color="red", annotation_text="80% target")
fig.update_layout(xaxis_tickformat=".0%")
show_plot(fig)


## Argument 8: Residual variance, entropy, and forecastability diagnostics support the information-limit conclusion

**How to read this.** These diagnostics are supporting evidence. They do not replace the business metric, but they help explain why the score ceiling is low.

- Residual variance ratio asks how much variance remains after the forecast.
- Analog conditional variance approximates irreducible uncertainty under the available feature state. For theory, see [Conditional expectations and smoothing](https://rafalab.dfci.harvard.edu/dsbook-part-2/ml/conditionals-and-smoothing.html).
- Residual diagnostics check whether obvious time-series signal remains. Forecast residuals should not contain usable autocorrelation; see [FPP3 residual diagnostics](https://otexts.com/fpp3/diagnostics.html).
- Spectral entropy summarizes forecastability as a process property; see Goerg's [Forecastable Component Analysis](https://proceedings.mlr.press/v28/goerg13.html).
- Entropy/Fano diagnostics translate remaining uncertainty in occurrence/spike/bucket tasks into classification accuracy limits; see this review of [Fano-style inequalities](https://pmc.ncbi.nlm.nih.gov/articles/PMC7516745/).
- Broader forecast evaluation caveats are discussed in [Forecast evaluation for data scientists: common pitfalls and best practices](https://pmc.ncbi.nlm.nih.gov/articles/PMC9718476/).


In [ ]:
def residual_variance_table(frame: pd.DataFrame) -> pd.DataFrame:
    work = business_penalty_frame(frame).copy()
    actual = work["TARGET_AMOUNT"].clip(lower=0.0)
    pred = work["PREDICTION"].clip(lower=0.0)
    residual = actual - pred
    var_actual = float(np.var(actual, ddof=0))
    var_residual = float(np.var(residual, ddof=0))
    return pd.DataFrame([{"RUN_NAME": work["RUN_NAME"].iloc[0], "VAR_ACTUAL": var_actual, "VAR_RESIDUAL": var_residual, "RESIDUAL_VARIANCE_RATIO": var_residual / var_actual if var_actual else np.nan, "EXPLAINED_VARIANCE_STYLE_SCORE": 1.0 - var_residual / var_actual if var_actual else np.nan}])


def analog_conditional_variance_summary(analog_frame: pd.DataFrame, base_frame: pd.DataFrame) -> pd.DataFrame:
    actual_var = float(np.var(pd.to_numeric(base_frame["TARGET_AMOUNT"], errors="coerce").fillna(0.0).clip(lower=0.0), ddof=0))
    valid = pd.to_numeric(analog_frame["ANALOG_TARGET_VARIANCE"], errors="coerce").dropna()
    expected_conditional_var = float(valid.mean()) if len(valid) else np.nan
    return pd.DataFrame([{"VAR_ACTUAL": actual_var, "ESTIMATED_E_VAR_Y_GIVEN_X": expected_conditional_var, "ESTIMATED_IRREDUCIBLE_VARIANCE_RATIO": expected_conditional_var / actual_var if actual_var else np.nan, "ESTIMATED_MAX_VARIANCE_EXPLAINED": 1.0 - expected_conditional_var / actual_var if actual_var else np.nan}])


def residual_diagnostic_table(frame: pd.DataFrame) -> pd.DataFrame:
    work = business_penalty_frame(frame).sort_values("FORECAST_DATE").copy()
    residual = work["PREDICTION"].clip(lower=0.0) - work["TARGET_AMOUNT"].clip(lower=0.0)
    rows = [{"TEST": "mean residual", "VALUE": float(residual.mean())}, {"TEST": "median residual", "VALUE": float(residual.median())}, {"TEST": "lag1 residual autocorrelation", "VALUE": float(residual.autocorr(lag=1))}, {"TEST": "lag7 residual autocorrelation", "VALUE": float(residual.autocorr(lag=7))}]
    if acorr_ljungbox is not None and len(work) > 30:
        lb = acorr_ljungbox(residual.fillna(0.0), lags=[7, 14, 28], return_df=True)
        for lag, row in lb.iterrows():
            rows.append({"TEST": f"Ljung-Box p-value lag {lag}", "VALUE": float(row["lb_pvalue"])})
    return pd.DataFrame(rows)


def residual_feature_mi_table(frame: pd.DataFrame) -> pd.DataFrame:
    if mutual_info_regression is None:
        return pd.DataFrame({"MESSAGE": ["scikit-learn mutual information is not available"]})
    work = business_penalty_frame(frame).merge(feature_frame[["FORECAST_DATE", *numeric_feature_columns]], on="FORECAST_DATE", how="left")
    X = work[numeric_feature_columns].apply(pd.to_numeric, errors="coerce").fillna(0.0)
    residual_abs = (work["PREDICTION"].clip(lower=0.0) - work["TARGET_AMOUNT"].clip(lower=0.0)).abs()
    mi = mutual_info_regression(X, residual_abs, random_state=0, n_neighbors=5)
    return pd.DataFrame({"FEATURE": numeric_feature_columns, "MI_WITH_ABS_RESIDUAL": mi}).sort_values("MI_WITH_ABS_RESIDUAL", ascending=False).reset_index(drop=True)


def spectral_entropy_forecastability(values: pd.Series) -> dict[str, float]:
    x = pd.to_numeric(values, errors="coerce").fillna(0.0).to_numpy(dtype="float64")
    x = x - x.mean()
    if len(x) < 4 or np.allclose(x, 0.0):
        return {"SPECTRAL_ENTROPY": np.nan, "FORECASTABILITY_OMEGA": np.nan}
    power = np.abs(np.fft.rfft(x)) ** 2
    power = power[1:]
    if power.sum() <= 0.0:
        return {"SPECTRAL_ENTROPY": np.nan, "FORECASTABILITY_OMEGA": np.nan}
    probs = power / power.sum()
    entropy = float(-(probs * np.log(probs + 1e-12)).sum() / np.log(len(probs)))
    return {"SPECTRAL_ENTROPY": entropy, "FORECASTABILITY_OMEGA": 1.0 - entropy}


def entropy_bits(values: pd.Series) -> float:
    probs = values.value_counts(normalize=True, dropna=False).to_numpy(dtype="float64")
    probs = probs[probs > 0]
    return float(-(probs * np.log2(probs)).sum())


def conditional_entropy_bits(y_class: pd.Series, signature: pd.Series) -> float:
    data = pd.DataFrame({"Y": y_class.astype("string"), "X": signature.astype("string")})
    total = len(data)
    return float(sum(len(group) / total * entropy_bits(group["Y"]) for _, group in data.groupby("X", dropna=False))) if total else np.nan


def binary_entropy(p: np.ndarray) -> np.ndarray:
    p = np.asarray(p, dtype="float64")
    out = np.zeros_like(p)
    mask = (p > 0) & (p < 1)
    out[mask] = -p[mask] * np.log2(p[mask]) - (1 - p[mask]) * np.log2(1 - p[mask])
    return out


def fano_error_lower_bound(h_cond: float, k_classes: int) -> float:
    if k_classes <= 1 or h_cond <= 0:
        return 0.0
    max_error = 1.0 - 1.0 / k_classes
    grid = np.linspace(0.0, max_error, 10001)
    rhs = binary_entropy(grid) + grid * math.log2(max(k_classes - 1, 1))
    feasible = grid[rhs >= h_cond]
    return float(feasible[0]) if len(feasible) else max_error


def available_state_signature(frame: pd.DataFrame) -> pd.Series:
    lag7_active = frame.get("TARGET_LAG_7", pd.Series(0.0, index=frame.index)).gt(0)
    lag28_active = frame.get("TARGET_LAG_28", pd.Series(0.0, index=frame.index)).gt(0)
    dow = frame.get("DAY_OF_WEEK", pd.Series("missing", index=frame.index)).astype(str)
    return dow + "|lag7=" + lag7_active.astype(str) + "|lag28=" + lag28_active.astype(str)


def entropy_fano_table(frame: pd.DataFrame) -> pd.DataFrame:
    y_amount = pd.to_numeric(frame["TARGET_AMOUNT"], errors="coerce").fillna(0.0).clip(lower=0.0)
    q90 = y_amount.quantile(0.90)
    buckets = pd.cut(y_amount, bins=[-0.1, 0.0, y_amount.quantile(0.50), y_amount.quantile(0.75), y_amount.quantile(0.90), np.inf], labels=["zero", "small", "medium", "large", "spike"], include_lowest=True)
    tasks = {"occurrence": y_amount.gt(0.0).map({True: "active", False: "zero"}), "spike_q90": y_amount.ge(q90).map({True: "spike", False: "normal"}), "amount_bucket": buckets.astype("string")}
    signature = available_state_signature(frame)
    rows = []
    for task, y_class in tasks.items():
        h_y = entropy_bits(y_class)
        h_cond = conditional_entropy_bits(y_class, signature)
        k = int(y_class.nunique(dropna=False))
        pe_lb = fano_error_lower_bound(h_cond, k)
        rows.append({"TASK": task, "CLASSES": k, "H_Y_BITS": h_y, "H_Y_GIVEN_AVAILABLE_STATE_BITS": h_cond, "MUTUAL_INFORMATION_BITS": h_y - h_cond, "FANO_ERROR_LOWER_BOUND": pe_lb, "FANO_ACCURACY_UPPER_BOUND": 1.0 - pe_lb})
    return pd.DataFrame(rows)


residual_variance = residual_variance_table(best_evidence_frame)
conditional_variance = analog_conditional_variance_summary(analog_bayes, feature_frame)
residual_diagnostics = residual_diagnostic_table(best_evidence_frame)
residual_mi = residual_feature_mi_table(best_evidence_frame)
spectral_summary = pd.DataFrame([spectral_entropy_forecastability(feature_frame["TARGET_AMOUNT"])])
entropy_fano = entropy_fano_table(feature_frame)

display(Markdown("### Residual and irreducible variance diagnostics"))
display(residual_variance)
display(conditional_variance)
display(residual_diagnostics)
display(Markdown("### Remaining residual feature signal"))
display(residual_mi.head(15))
display(Markdown("### Forecastability and entropy diagnostics"))
display(spectral_summary)
display(entropy_fano)

fig = px.bar(residual_mi.head(15), x="FEATURE", y="MI_WITH_ABS_RESIDUAL", title="Mutual information between available features and absolute residuals")
fig.update_layout(xaxis_tickangle=-35)
show_plot(fig)

fig = px.bar(entropy_fano, x="TASK", y="FANO_ACCURACY_UPPER_BOUND", title="Fano-style upper bound on classification accuracy by subproblem")
fig.update_layout(yaxis_tickformat=".0%")
show_plot(fig)

fig = go.Figure(go.Indicator(mode="gauge+number", value=float(spectral_summary["FORECASTABILITY_OMEGA"].iloc[0]), title={"text": "Spectral forecastability omega"}, gauge={"axis": {"range": [0, 1]}}))
show_plot(fig)


## Evidence Summary

This final table gathers the headline evidence. The strongest professional conclusion is supported when deployable models, impossible hindsight selectors, optimistic analog ceiling estimates, and supporting diagnostics remain materially below the 80% target.


In [ ]:
summary_rows = [
    {"EVIDENCE": "best simple point-in-time candidate", "SCORE": best_simple_score},
    {"EVIDENCE": "simple candidate hindsight oracle", "SCORE": weighted_business_score(simple_oracle["TARGET_AMOUNT"], simple_oracle["PREDICTION"])},
]
if not external_scores.empty:
    summary_rows.append({"EVIDENCE": f"best external/saved model: {external_scores.iloc[0]['RUN_NAME']}", "SCORE": float(external_scores.iloc[0]["WEIGHTED_BUSINESS_SCORE"])})
if not model_library_oracle.empty:
    summary_rows.append({"EVIDENCE": "model-library hindsight oracle", "SCORE": weighted_business_score(model_library_oracle["TARGET_AMOUNT"], model_library_oracle["PREDICTION"])})
summary_rows.extend(
    [
        {"EVIDENCE": "analog Bayes-style score optimizer", "SCORE": analog_bayes_score},
        {"EVIDENCE": "analog Bayes upper bootstrap CI", "SCORE": float(bootstrap_ci.loc["UPPER_95"])},
        {"EVIDENCE": "client target", "SCORE": CLIENT_TARGET_SCORE},
    ]
)
evidence_summary = pd.DataFrame(summary_rows)
evidence_summary["GAP_TO_80"] = evidence_summary["SCORE"] - CLIENT_TARGET_SCORE
display(evidence_summary)

fig = px.bar(evidence_summary.dropna(subset=["SCORE"]).sort_values("SCORE"), x="SCORE", y="EVIDENCE", orientation="h", title="Final TRF+ attainability evidence ladder")
fig.add_vline(x=CLIENT_TARGET_SCORE, line_dash="dash", line_color="red", annotation_text="80% target")
show_plot(fig)

print("Notebook completed.")
